# 11 — Why Infeasibility Arises in GA for TSP

**Maps to:** `report/Chapters/Task2.tex` §`T2:Infeasibility`.  
**Ticket:** TICKET-11.

Walk through why a permutation-encoded GA for the TSP produces infeasible offspring. The primary GA pipeline (notebook 10, notebook 15) uses **naive single-point crossover**, so infeasibility is generated in every generation and the repair mechanism in notebook 12 is what converts those children back into valid tours.

This notebook works the problem from both sides: a worked example showing how naive crossover breaks the permutation property, then three permutation-aware operators (PMX, OX, CX) that demonstrate the alternative — feasibility preservation by construction. The contrast motivates the repair mechanism and frames the constraint-handling comparison in notebook 17 (repair vs penalty vs PMX). Pure prose + small code demos — no new module code.

## Analytical Section: Why Infeasibility Arises

In the TSP representation used in this project, a chromosome is feasible only if it is a permutation of the city set. For an $n$-city instance, every chromosome must contain each city in $\\{0,1,\\ldots,n-1\\}$ exactly once. The fitness function then interprets the chromosome as a tour by summing the distance from each city to the next city, plus the return edge from the last city to the first.

Infeasibility arises when a crossover operator treats a TSP chromosome like an ordinary fixed-position string. A standard one-point, two-point, or uniform crossover copies values position by position without checking whether the copied values have already appeared elsewhere in the child. That is harmless for binary strings, but not for permutations. If a city is copied twice, then some other city must be absent, because the chromosome length remains fixed. The child still has $n$ entries, but it no longer describes a Hamiltonian tour.

The example below uses the same parents and cut points as `08_crossover.ipynb` so that the infeasible naive crossover can be compared directly with PMX, OX, and CX.


In [1]:
import numpy as np

parent_a = np.array([0, 1, 2, 3, 4, 5, 6, 7])
parent_b = np.array([2, 6, 4, 0, 5, 7, 1, 3])
pt1, pt2 = 2, 4
n_cities = len(parent_a)

def is_valid_permutation(chromosome, n):
    return sorted(chromosome.tolist()) == list(range(n))

def duplicate_cities(chromosome):
    return sorted({int(x) for x in chromosome if chromosome.tolist().count(int(x)) > 1})

def missing_cities(chromosome, n):
    return sorted(set(range(n)) - set(map(int, chromosome)))

print(f"Parent A: {parent_a}")
print(f"Parent B: {parent_b}")
print(f"Cut segment: positions {pt1}..{pt2}, Parent A values {parent_a[pt1:pt2+1]}")


Parent A: [0 1 2 3 4 5 6 7]
Parent B: [2 6 4 0 5 7 1 3]
Cut segment: positions 2..4, Parent A values [2 3 4]


## Naive Two-Point Crossover: How Feasibility Breaks

A naive two-point crossover starts with Parent B and blindly overwrites the cut segment with the segment from Parent A. With cut positions $2..4$, the copied segment is $[2,3,4]$.

$$
\begin{align*}
A &= [0,1,\boxed{2,3,4},5,6,7] \\
B &= [2,6,\boxed{4,0,5},7,1,3]
\end{align*}
$$

The overwrite creates the child

$$
[2,6,\boxed{2,3,4},7,1,3]
$$

The step-by-step cause of infeasibility is:

| Position | Original value from Parent B | After overwrite | Consequence |
|---:|---:|---:|---|
| 2 | 4 | 2 | city 2 is inserted, while city 4 remains present at position 4 |
| 3 | 0 | 3 | city 0 is removed, and city 3 is inserted |
| 4 | 5 | 4 | city 5 is removed, and city 4 is inserted |

After the overwrite, city `2` appears at positions `0` and `2`, and city `3` appears at positions `3` and `7`. Cities `0` and `5` disappear. This child has the correct length, but it is not a valid TSP tour because it visits two cities twice and never visits two other cities.


In [6]:
def naive_two_point_crossover(parent_a, parent_b, pt1, pt2):
    child = parent_b.copy()
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    return child

child_naive = naive_two_point_crossover(parent_a, parent_b, pt1, pt2)

print(f"Naive child: {child_naive}")
print(f"Duplicates:  {duplicate_cities(child_naive)}")
print(f"Missing:     {missing_cities(child_naive, n_cities)}")
print(f"Valid:       {is_valid_permutation(child_naive, n_cities)}")


Naive child: [2 6 2 3 4 7 1 3]
Duplicates:  [2, 3]
Missing:     [0, 5]
Valid:       False


This is the main source of crossover infeasibility in a permutation-based GA for the TSP: the operator preserves positions but not the set constraint. The same problem can occur with any crossover that copies genes independently without a permutation repair step.

## PMX: Why the Mapping Chain Preserves Feasibility

Partially Mapped Crossover (PMX) is permutation-aware. It also copies the selected segment from Parent A, but unlike naive crossover, it resolves conflicts using the position-wise mapping between the two parent segments.

For the same cut segment:

$$
A_{2..4}=[2,3,4], \quad B_{2..4}=[4,0,5].
$$

These segment positions define mappings between Parent A and Parent B values:

$$
2 \leftrightarrow 4, \quad 3 \leftrightarrow 0, \quad 4 \leftrightarrow 5
$$

PMX first copies Parent A's segment into the child:

$$
[-,-,2,3,4,-,-,-]
$$

It then considers the Parent B segment values. Value `4` is already present in the copied segment, so it is skipped. Value `0` is not present. Since `0` came from position 3, PMX follows the mapping through Parent A's value at that position: `3`. City `3` is found in Parent B at position 7, which is outside the copied segment, so `0` is placed at position 7. Value `5` is not present. It maps through Parent A value `4`; Parent B has `4` at position 2, which is still inside the segment, so the chain continues through Parent A value `2`. Parent B has `2` at position 0, outside the segment, so `5` is placed at position 0.

The remaining empty positions are filled directly from Parent B. The result is feasible because each conflict is redirected to a unique outside position instead of being copied on top of an existing city.


In [7]:
def pmx(parent_a, parent_b, pt1, pt2):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    segment_vals = set(child[pt1:pt2 + 1])

    for i in range(pt1, pt2 + 1):
        val = parent_b[i]
        if val not in segment_vals:
            j = i
            while pt1 <= j <= pt2:
                j = np.where(parent_b == parent_a[j])[0][0]
            child[j] = val

    empty = child == -1
    child[empty] = parent_b[empty]
    return child

child_pmx = pmx(parent_a, parent_b, pt1, pt2)
print(f"PMX child: {child_pmx}")
print(f"Duplicates: {duplicate_cities(child_pmx)}")
print(f"Missing:    {missing_cities(child_pmx, n_cities)}")
print(f"Valid:      {is_valid_permutation(child_pmx, n_cities)}")


PMX child: [5 6 2 3 4 7 1 0]
Duplicates: []
Missing:    []
Valid:      True


PMX therefore does not break the permutation property when implemented correctly. Its mapping chain gives every displaced value one legal destination, so duplicates and missing cities are avoided by construction.

## OX: Why Ordered Filling Preserves Feasibility

Order Crossover (OX) preserves a segment from Parent A and fills the remaining positions using the order in Parent B. The important difference from naive crossover is that OX skips any value already copied into the child.

First, OX copies the Parent A segment:

$$
[-,-,2,3,4,-,-,-]
$$

The copied values are `2`, `3`, and `4`. Starting immediately after the second cut point and wrapping around Parent B, OX scans Parent B in this order:

$$
7,1,3,2,6,4,0,5
$$

It skips values already in the segment (`3`, `2`, and `4`) and keeps the remaining values:

$$
[7,1,6,0,5]
$$

Those five values are exactly the five missing cities, so they fill the five empty positions from position 5 onward, with wrap-around:

$$
[6,0,2,3,4,7,1,5]
$$

OX is feasible because the child is built from two disjoint sets: the copied segment and the complement collected from Parent B. No collected value can duplicate a segment value because duplicates are explicitly skipped.


In [4]:
def ox(parent_a, parent_b, pt1, pt2):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    segment_vals = set(child[pt1:pt2 + 1])

    remaining = []
    for i in range(n):
        idx = (pt2 + 1 + i) % n
        if parent_b[idx] not in segment_vals:
            remaining.append(parent_b[idx])

    j = 0
    for i in range(n):
        idx = (pt2 + 1 + i) % n
        if child[idx] == -1:
            child[idx] = remaining[j]
            j += 1

    return child

child_ox = ox(parent_a, parent_b, pt1, pt2)
print(f"OX child: {child_ox}")
print(f"Duplicates: {duplicate_cities(child_ox)}")
print(f"Missing:    {missing_cities(child_ox, n_cities)}")
print(f"Valid:      {is_valid_permutation(child_ox, n_cities)}")


OX child: [0 5 2 3 4 7 1 6]
Duplicates: []
Missing:    []
Valid:      True


OX therefore preserves feasibility while still recombining useful structure. It keeps one parent's local segment and the other parent's relative ordering for the remaining cities.

## CX: Why Cycle Discovery Preserves Feasibility

Cycle Crossover (CX) uses the position-wise relationship between the two parents. It identifies cycles of indices by starting at one position, reading the value from Parent B, finding where that value occurs in Parent A, and repeating until the cycle returns to the starting position.

For the same parents, the first cycle starts at position 0:

| Step | Current position | Parent B value | Position of that value in Parent A |
|---:|---:|---:|---:|
| 1 | 0 | 2 | 2 |
| 2 | 2 | 4 | 4 |
| 3 | 4 | 5 | 5 |
| 4 | 5 | 7 | 7 |
| 5 | 7 | 3 | 3 |
| 6 | 3 | 0 | 0 |

This returns to the starting position, so the first cycle is positions `0, 2, 4, 5, 7, 3`. The remaining unvisited positions form a second cycle: `1, 6`.

CX then alternates which parent supplies each cycle. If cycle 1 is taken from Parent A and cycle 2 from Parent B, the child is:

$$
[0,6,2,3,4,5,1,7]
$$

This remains feasible because cycles partition the set of positions. Within a cycle, taking values from one parent gives exactly the same set of city values that the other parent maps around that cycle. Across all cycles, every position is assigned once and every city appears once.


In [5]:
def cx(parent_a, parent_b):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    visited = np.zeros(n, dtype=bool)
    pos_in_a = {val: idx for idx, val in enumerate(parent_a)}

    cycle_num = 0
    cycles = []
    for start in range(n):
        if visited[start]:
            continue

        cycle_indices = []
        i = start
        while not visited[i]:
            visited[i] = True
            cycle_indices.append(i)
            i = pos_in_a[parent_b[i]]

        source = parent_a if cycle_num % 2 == 0 else parent_b
        for idx in cycle_indices:
            child[idx] = source[idx]

        cycles.append(cycle_indices)
        cycle_num += 1

    return child, cycles

child_cx, cycles = cx(parent_a, parent_b)
print(f"CX cycles: {cycles}")
print(f"CX child:  {child_cx}")
print(f"Duplicates: {duplicate_cities(child_cx)}")
print(f"Missing:    {missing_cities(child_cx, n_cities)}")
print(f"Valid:      {is_valid_permutation(child_cx, n_cities)}")


CX cycles: [[0, 2, 4, 5, 7, 3], [1, 6]]
CX child:  [0 6 2 3 4 5 1 7]
Duplicates: []
Missing:    []
Valid:      True


CX therefore also preserves the permutation property when implemented correctly. Its cycle discovery step ensures that whole position cycles are inherited, rather than mixing individual positions in a way that could duplicate a city.

## Consequences for Fitness Evaluation and Search

Infeasible chromosomes are harmful because their fitness values do not represent valid TSP tours. If a chromosome repeats a city, the tour-length calculation may include edges into and out of that repeated city while completely ignoring a missing city. This can make the computed distance artificially short or otherwise incomparable with feasible tours. The GA selection step then treats this meaningless number as if it were a legitimate tour cost.

Once infeasible chromosomes enter the population, selection pressure becomes corrupted. A child with duplicate cities may appear fitter than a valid child simply because it skipped expensive cities. Tournament or roulette selection can then favour the infeasible child, allowing invalid genetic material to spread. Crossover between infeasible parents can create more severe violations, and the search may converge toward chromosomes that minimise an invalid objective rather than toward high-quality Hamiltonian tours.

This is exactly why the primary GA pipeline pairs naive single-point crossover with the repair mechanism developed in notebook 12. The repair step runs on every offspring before fitness evaluation: it detects duplicates, identifies missing cities, and replaces repeated entries so that the chromosome is converted back into a valid permutation. Without repair, naive crossover would flood the population with invalid tours within the first few generations and the corrupted selection pressure described above would take over. Repair is not a safeguard tacked on around an otherwise-clean operator; it is the production constraint-handling step that makes naive crossover viable as the primary recombination operator. PMX, OX, and CX remain available as feasibility-preserving alternatives — notebook 17 compares the repair-based pipeline against those alternatives (and against a penalty-function approach) to address §3.3 of the spec.

## Summary

| Operator | Breaks permutation property? | Mechanism | Worked-example outcome |
|---|---:|---|---|
| Naive two-point crossover | Yes | Blindly overwrites positions without checking duplicates | `[2,6,2,3,4,7,1,3]`; duplicates `2,3`; missing `0,5` |
| PMX | No | Mapping chain redirects displaced values to unique outside positions | `[5,6,2,3,4,7,1,0]`; valid permutation |
| OX | No | Ordered fill skips values already copied in the segment | `[6,0,2,3,4,7,1,5]`; valid permutation |
| CX | No | Cycles partition positions and inherit whole cycles from parents | `[0,6,2,3,4,5,1,7]`; valid permutation |

Infeasibility in a GA for the TSP is not caused by permutation-aware crossovers such as PMX, OX, and CX themselves. It is caused by applying general-purpose crossover logic to a permutation representation. The practical consequence is that the GA must either use operators that preserve the permutation property or apply repair before evaluating and selecting offspring. The primary pipeline in this project takes the second route — naive crossover plus repair — so that the repair mechanism is the load-bearing constraint-handling step the assignment's §2.2 and §3.3 call out.
